In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# For sentence-transformers
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import torch
import datetime

# Paths
DATA_PROCESSED = Path('../data/processed')
MODEL_DIR = Path('../models/regmap-embedder')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

C:\Users\stamp\AppData\Local\Temp\ipykernel_21612\2878646266.py:8: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
C:\Users\stamp\AppData\Local\Temp\ipykernel_21612\2878646266.py:8: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation


In [7]:
# Define DATA_RAW (same as in data prep notebook)
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

In [11]:
# Reload the raw crosswalk
crosswalk_raw = pd.read_csv(DATA_RAW / 'nist_800_53_rev5_hipaa_crosswalk.csv')

# Clean column names (remove newlines, collapse spaces)
crosswalk_raw.columns = (
    crosswalk_raw.columns
    .str.replace('\n', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Rename to our standard names
crosswalk = crosswalk_raw.rename(columns={
    'Focal Document Element': 'nist_control_id',
    'Focal Document Element Description': 'nist_text',
    'Reference Document Element': 'hipaa_citation',
    'Reference Document Element Description': 'hipaa_text'
})

# Keep only needed columns
crosswalk = crosswalk[['nist_control_id', 'nist_text', 'hipaa_citation', 'hipaa_text']]

# Drop rows with missing text
crosswalk.dropna(subset=['nist_text', 'hipaa_text', 'hipaa_citation'], inplace=True)

# Strip whitespace
for col in ['nist_text', 'hipaa_text', 'hipaa_citation']:
    crosswalk[col] = crosswalk[col].astype(str).str.strip()

# Remove exact duplicates
crosswalk = crosswalk.drop_duplicates(subset=['nist_control_id', 'hipaa_citation'])

# All remaining rows are true mappings (label = 1)
crosswalk['label'] = 1

print(f"Positive examples: {len(crosswalk)}")
crosswalk.head()

Positive examples: 40


,nist_control_id,nist_text,hipaa_citation,hipaa_text,label
0,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3),Workforce Security: Implement policies and pro...,1
1,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3)(ii)(A),Authorization and/or Supervision (A): Implemen...,1
2,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4),Information Access Management: Implement polic...,1
3,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(B),Access Authorization (A): Implement policies a...,1
4,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(C),Access Establishment and Modification (A): Imp...,1


In [12]:
import random
random.seed(SEED)

# Unique control IDs and HIPAA citations
nist_ids = crosswalk['nist_control_id'].unique()
hipaa_citations = crosswalk['hipaa_citation'].unique()

# Map each NIST control to its known positive HIPAA citations
positive_map = {}
for _, row in crosswalk.iterrows():
    positive_map.setdefault(row['nist_control_id'], set()).add(row['hipaa_citation'])

# Generate negative pairs (random NIST + HIPAA that are not known positives)
# We'll aim for 3 negatives per positive
num_negatives_target = len(crosswalk) * 3
negative_pairs = []
attempts = 0
max_attempts = num_negatives_target * 10  # safety

while len(negative_pairs) < num_negatives_target and attempts < max_attempts:
    nist_id = random.choice(nist_ids)
    neg_hipaa = random.choice(hipaa_citations)
    if neg_hipaa not in positive_map.get(nist_id, set()):
        negative_pairs.append({'nist_control_id': nist_id, 'hipaa_citation': neg_hipaa, 'label': 0})
    attempts += 1

print(f"Negative pairs generated: {len(negative_pairs)}")

# Convert to DataFrame
neg_df = pd.DataFrame(negative_pairs)

# Merge with texts
nist_texts = crosswalk[['nist_control_id', 'nist_text']].drop_duplicates()
hipaa_texts = crosswalk[['hipaa_citation', 'hipaa_text']].drop_duplicates()
neg_df = neg_df.merge(nist_texts, on='nist_control_id', how='inner')
neg_df = neg_df.merge(hipaa_texts, on='hipaa_citation', how='inner')

# Combine positives and negatives
full_df = pd.concat([crosswalk, neg_df], ignore_index=True)
# Shuffle
full_df = full_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Full dataset: {len(full_df)} rows")
print("Label distribution:\n", full_df['label'].value_counts())

# Save
full_df.to_csv(DATA_PROCESSED / 'training_pairs.csv', index=False)
print("Saved training_pairs.csv")

Negative pairs generated: 120
Full dataset: 160 rows
Label distribution:
 label
0    120
1     40
Name: count, dtype: int64
Saved training_pairs.csv


In [13]:
# Load the balanced training dataset
full_df = pd.read_csv(DATA_PROCESSED / 'training_pairs.csv')
print(f"Loaded {len(full_df)} pairs")
print(full_df['label'].value_counts())

Loaded 160 pairs
label
0    120
1     40
Name: count, dtype: int64


In [14]:
labeled = pd.read_csv(DATA_PROCESSED / 'labeled_pairs.csv')
print(f"Total pairs: {len(labeled)}")
print("Label distribution:\n", labeled['label'].value_counts())
labeled.head(3)

Total pairs: 40
Label distribution:
 label
0    40
Name: count, dtype: int64


,nist_control_id,nist_text,hipaa_citation,hipaa_text,fulfilled_by,strength_of_relationship,label
0,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3),Workforce Security: Implement policies and pro...,NaN,NaN,0
1,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3)(ii)(A),Authorization and/or Supervision (A): Implemen...,NaN,NaN,0
2,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4),Information Access Management: Implement polic...,NaN,NaN,0


In [15]:
print(labeled.isnull().sum())
# If there are null texts, drop them
labeled.dropna(subset=['nist_text', 'hipaa_text'], inplace=True)

nist_control_id              0
nist_text                    0
hipaa_citation               0
hipaa_text                   0
fulfilled_by                40
strength_of_relationship    40
label                        0
dtype: int64


In [16]:
# Combine control and HIPAA text into a single feature for stratification?
# We'll stratify by the label column directly.
train_val, test = train_test_split(
    labeled,
    test_size=0.15,
    random_state=SEED,
    stratify=labeled['label']
)
train, val = train_test_split(
    train_val,
    test_size=0.15,  # ~15% of total for val
    random_state=SEED,
    stratify=train_val['label']
)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print("Train label dist:\n", train['label'].value_counts())
print("Val label dist:\n", val['label'].value_counts())
print("Test label dist:\n", test['label'].value_counts())

Train: 28 | Val: 6 | Test: 6
Train label dist:
 label
0    28
Name: count, dtype: int64
Val label dist:
 label
0    6
Name: count, dtype: int64
Test label dist:
 label
0    6
Name: count, dtype: int64


In [17]:
def df_to_examples(df):
    examples = []
    for _, row in df.iterrows():
        examples.append(
            InputExample(texts=[row['nist_text'], row['hipaa_text']], label=float(row['label']))
        )
    return examples

train_examples = df_to_examples(train)
val_examples = df_to_examples(val)
test_examples = df_to_examples(test)

print(f"Train examples: {len(train_examples)}")

Train examples: 28


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.ContrastiveLoss(model=model, margin=0.5)

In [ ]:
# Build validation evaluator using InformationRetrievalEvaluator
# It requires: queries (NIST texts), corpus (all HIPAA texts), and relevant_docs (mapping of query id -> set of corpus ids that are relevant).

from sentence_transformers.evaluation import InformationRetrievalEvaluator

# For validation, we need to build a mini-corpus of unique HIPAA texts
val_hipaa_texts = val['hipaa_text'].unique()
corpus = {i: text for i, text in enumerate(val_hipaa_texts)}
corpus_texts = list(corpus.values())

# Map each NIST control to its relevant HIPAA document ids
queries = {}
relevant_docs = {}
query_idx = 0
for _, row in val.iterrows():
    nist_text = row['nist_text']
    hipaa_text = row['hipaa_text']
    # Find the corpus id of this HIPAA text
    # (there might be duplicates, we'll take the first match)
    corpu_id = None
    for idx, ctext in corpus.items():
        if ctext == hipaa_text:
            corpu_id = idx
            break
    if corpu_id is None:
        continue
    # Add query (NIST text) to queries dict
    queries[query_idx] = nist_text
    # Add relevance: only this HIPAA citation is relevant (label=1)
    if row['label'] == 1:
        relevant_docs[query_idx] = {corpu_id}
    else:
        # For negative examples, we still use the paired HIPAA as the "relevant" for evaluation
        # (but it's actually not mapped; we'll handle that by ignoring queries with no positive)
        # Better: only include queries that have at least one positive mapping.
        pass  # We'll skip negative-only queries for evaluation
    query_idx += 1

# Filter out queries with no positive (if any)
queries_filtered = {}
relevant_docs_filtered = {}
for qid in queries:
    if qid in relevant_docs and len(relevant_docs[qid]) > 0:
        queries_filtered[qid] = queries[qid]
        relevant_docs_filtered[qid] = relevant_docs[qid]

print(f"Validation queries (with positive mappings): {len(queries_filtered)}")

# We also need to provide a mapping from query IDs to true relevant doc IDs, used for recall@k
# Create the evaluator
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries_filtered,
    corpus=corpus,
    relevant_docs=relevant_docs_filtered,
    name="val-hipaa",
    show_progress_bar=True
)

In [ ]:
BATCH_SIZE = 16  # adjust based on your GPU/CPU memory
EPOCHS = 10
WARMUP_STEPS = int(len(train_examples) * 0.1)

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    evaluator=ir_evaluator,
    evaluation_steps=500,  # run evaluation every 500 training steps
    output_path=str(MODEL_DIR),
    save_best_model=True,
    show_progress_bar=True
)

In [ ]:
BATCH_SIZE = 16  # adjust based on your GPU/CPU memory
EPOCHS = 10
WARMUP_STEPS = int(len(train_examples) * 0.1)

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    evaluator=ir_evaluator,
    evaluation_steps=500,  # run evaluation every 500 training steps
    output_path=str(MODEL_DIR),
    save_best_model=True,
    show_progress_bar=True
)

In [ ]:
# Build test corpus (all HIPAA texts in test)
test_corpus = {i: text for i, text in enumerate(test['hipaa_text'].unique())}
test_corpus_list = list(test_corpus.values())
corpus_embeddings = best_model.encode(test_corpus_list, convert_to_tensor=True)

# For each test query with label=1, compute rank of its true HIPAA
recall_1, recall_3, recall_5 = 0, 0, 0
mrr = 0
total_queries = 0

for _, row in test.iterrows():
    if row['label'] != 1:
        continue  # only evaluate on positive mappings
    query_text = row['nist_text']
    true_hipaa = row['hipaa_text']
    # Find corpus id of true HIPAA
    true_id = None
    for cid, ctext in test_corpus.items():
        if ctext == true_hipaa:
            true_id = cid
            break
    if true_id is None:
        continue
    query_embed = best_model.encode(query_text, convert_to_tensor=True)
    cos_scores = torch.nn.functional.cosine_similarity(query_embed, corpus_embeddings)
    # Rank indices by descending score
    sorted_indices = torch.argsort(cos_scores, descending=True)
    rank = (sorted_indices == true_id).nonzero(as_tuple=True)[0].item() + 1
    if rank <= 1:
        recall_1 += 1
    if rank <= 3:
        recall_3 += 1
    if rank <= 5:
        recall_5 += 1
    mrr += 1 / rank
    total_queries += 1

recall_1 /= total_queries
recall_3 /= total_queries
recall_5 /= total_queries
mrr /= total_queries

print(f"Test queries evaluated: {total_queries}")
print(f"Recall@1: {recall_1:.4f}")
print(f"Recall@3: {recall_3:.4f}")
print(f"Recall@5: {recall_5:.4f}")
print(f"MRR: {mrr:.4f}")